# HLTV

In [ ]:
# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/ranking/teams/")
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': datetime.now().strftime('%Y-%m-%d'),
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings)

# Futtasd hetente → time-series ranking data
rankings = scrape_team_rankings()
display(rankings)

In [ ]:
# Events

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_major_events():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/events/archive?eventType=MAJOR")
    
    # Várunk, amíg betöltődnek az események hónapjai
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "events-month")))
    
    events_data = []
    months_divs = driver.find_elements(By.CLASS_NAME, "events-month")
    
    for month_div in months_divs:
        try:
            # Hónap
            month_name = month_div.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Minden esemény az adott hónapban
            event_links = month_div.find_elements(By.CSS_SELECTOR, "a.small-event.standard-box")
            
            for event in event_links:
                try:
                    event_name = event.find_element(By.CSS_SELECTOR, ".event-col .text-ellipsis").text.strip()
                    team_count = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.small-col")[0].text.strip()
                    prize = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.prizePoolEllipsis")[0].get_attribute("title").strip()
                    link = event.get_attribute("href").strip()
                    event_id = link.split('/')[4]
                    
                    events_data.append({
                        "month": month_name,
                        "event_name": event_name,
                        "event_id": event_id,
                        "teams": team_count,
                        "prize": prize,
                        "link": link
                    })
                except Exception as e_event:
                    print(f"Esemény hiba: {e_event}")
                    continue
        except Exception as e_month:
            print(f"Hónap hiba: {e_month}")
            continue
    
    driver.quit()
    return pd.DataFrame(events_data)

# Futtatás
major_events = scrape_major_events()
display(major_events.sample(5))

In [ ]:
# Matches of event

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_event_results(event_id):
    url = f"https://www.hltv.org/results?event={event_id}"
    driver = webdriver.Chrome()
    driver.get(url)
    
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "results-sublist")))
    
    results_data = []
    sublists = driver.find_elements(By.CLASS_NAME, "results-sublist")
    
    for sublist in sublists:
        try:
            # Mérkőzés dátuma
            match_date = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Az összes mérkőzés az adott dátumban
            matches = sublist.find_elements(By.CSS_SELECTOR, ".result-con a")
            
            for match in matches:
                try:
                    link = match.get_attribute("href").strip()
                    
                    team_home = match.find_element(By.CSS_SELECTOR, ".team1 .team").text.strip()
                    team_away = match.find_element(By.CSS_SELECTOR, ".team2 .team").text.strip()
                    
                    # Pontok a sorrend alapján
                    score_spans = match.find_elements(By.CSS_SELECTOR, ".result-score span")
                    score_home = int(score_spans[0].text.strip())
                    score_away = int(score_spans[1].text.strip())
                    
                    map_type = match.find_element(By.CSS_SELECTOR, ".map-and-stars .map-text").text.strip()
                    
                    rounds = int(map_type[-1]) if map_type[:2] == "bo" else 1

                    results_data.append({
                        "date": match_date,
                        "team_home": team_home,
                        "team_away": team_away,
                        "score_home": score_home,
                        "score_away": score_away,
                        "map": map_type,
                        "rounds": rounds,
                        "link": link
                    })
                except Exception as e_match:
                    print(f"Mérkőzés hiba: {e_match}")
                    continue
        except Exception as e_sublist:
            print(f"Dátum hiba: {e_sublist}")
            continue
    
    driver.quit()
    return pd.DataFrame(results_data)

# Példa
event_results = scrape_event_results(7902)
display(event_results.sample(5))


In [12]:
# Match - Head-to-Head, Stats

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import numpy as np
import time

def scrape_head_to_head(match_url):
    driver = webdriver.Chrome()
    driver.get(match_url)
    wait = WebDriverWait(driver, 15)

    data = []

    try:
        # --- HEAD TO HEAD rész ---
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "head-to-head")))
        h2h_box = driver.find_element(By.CLASS_NAME, "head-to-head")
        container = h2h_box.find_element(By.CLASS_NAME, "standard-box")

        team1 = container.find_element(By.CSS_SELECTOR, ".team1 .teamName").text.strip()
        team2 = container.find_element(By.CSS_SELECTOR, ".team2 .teamName").text.strip()
        
        mapholders = driver.find_elements(By.CLASS_NAME, "mapholder")

        map_data = []
        for mapholder in mapholders:
            try:
                mapname = mapholder.find_element(By.CLASS_NAME, "mapname").text.strip()
                
                results_div = mapholder.find_element(By.XPATH, ".//*[contains(@class, 'results ')]")
                results_class = results_div.get_attribute("class")
                played = 'played' in results_class

                if not played:
                    map_data.append({
                        "map_name": mapname,
                        "played": played,
                        "home_win": None,
                        "home_pick": None,
                        "home_score": -1,
                        "away_score": -1
                    })
                    continue

                # Bal és jobb oldal div-ek dinamikus classokkal
                left_div = mapholder.find_element(By.XPATH, ".//*[contains(@class, 'results-left')]")
                right_div = mapholder.find_element(By.XPATH, ".//*[contains(@class, 'results-right')]")

                # class attribútum alapján állapotok (won / lost / pick)
                left_classes = left_div.get_attribute("class")

                left_win = "win" if "won" in left_classes else "loss" if "lost" in left_classes else "unknown"

                left_pick = "pick" in left_classes

                # score kiolvasás
                left_score = left_div.find_element(By.CLASS_NAME, "results-team-score").text
                right_score = right_div.find_element(By.CLASS_NAME, "results-team-score").text

                map_data.append({
                    "map_name": mapname,
                    "played": played,
                    "home_win": left_win,
                    "home_pick": left_pick,
                    "home_score": left_score,
                    "away_score": right_score
                })

            except Exception as e:
                print(f"⚠️ Hiba egy map feldolgozásánál: {e}")
                continue

        import json

        # --- Map-level feature aggregálás ---
        if map_data:
            maps_played = sum(1 for m in map_data if m["played"])
            maps_won = sum(1 for m in map_data if m["home_win"] == "win")
            maps_lost = sum(1 for m in map_data if m["home_win"] == "loss")
            maps_picked_by_home = sum(1 for m in map_data if m["home_pick"])
            avg_score_diff = np.mean([
                int(m["home_score"]) - int(m["away_score"])
                for m in map_data if m["played"]
            ]) if maps_played > 0 else None

            # Map winrate (arány)
            map_winrate = maps_won / maps_played if maps_played > 0 else None

            # JSON formában is tároljuk, ha később map-szinten akarjuk kinyerni
            map_json = json.dumps(map_data, ensure_ascii=False)

        else:
            maps_played = maps_won = maps_lost = maps_picked_by_home = 0
            avg_score_diff = map_winrate = None
            map_json = "[]"

        stats = container.find_elements(By.CSS_SELECTOR, ".flexbox-column.grow .bold")
        wins_team1 = int(stats[0].text.strip())
        overtimes = int(stats[1].text.strip())
        wins_team2 = int(stats[2].text.strip())
        total_non_ot = wins_team1 + wins_team2
        home_win_rate = wins_team1 / total_non_ot if total_non_ot > 0 else None

        # --- PLAYER STAT rész ---
        # Várunk, hogy a statisztika box betöltődjön
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "stats-content")))
        time.sleep(2)

        stats_tables = driver.find_elements(By.CSS_SELECTOR, "table.totalstats")

        def parse_team_stats(table):
            """Parseolja a team táblát és visszaadja a statok listáját."""
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]  # első sor a header
            team_name = table.find_element(By.CSS_SELECTOR, ".teamName.team").text.strip()

            ratings, adrs, swings = [], [], []

            for r in rows:
                try:
                    rating = float(r.find_element(By.CSS_SELECTOR, ".rating").text.strip())
                    adr = float(r.find_element(By.CSS_SELECTOR, ".adr").text.strip())
                    swing_text = r.find_element(By.CSS_SELECTOR, ".roundSwing").text.strip().replace('%','')
                    swing = float(swing_text.replace('+', '').replace(',', '.'))
                    ratings.append(rating)
                    adrs.append(adr)
                    swings.append(swing)
                except Exception:
                    continue

            return team_name, ratings, adrs, swings

        # Feltételezzük, hogy az első totalstats a home team (team1), a második az away (team2)
        home_team_name, home_ratings, home_adrs, home_swings = parse_team_stats(stats_tables[0])
        away_team_name, away_ratings, away_adrs, away_swings = parse_team_stats(stats_tables[1])

        data.append({
            "home_team": team1,
            "away_team": team2,
            "wins_home": wins_team1,
            "wins_away": wins_team2,
            "overtimes": overtimes,
            "total_non_overtime": total_non_ot,
            "home_win_rate": round(home_win_rate, 4) if home_win_rate is not None else None,
            "home_team_avg_rating": np.mean(home_ratings) if home_ratings else None,
            "home_team_std_rating": np.std(home_ratings) if home_ratings else None,
            "home_team_avg_ADR": np.mean(home_adrs) if home_adrs else None,
            "home_team_std_ADR": np.std(home_adrs) if home_adrs else None,
            "home_team_avg_Swing": np.mean(home_swings) if home_swings else None,
            "home_team_std_Swing": np.std(home_swings) if home_swings else None,
            "away_team_avg_rating": np.mean(away_ratings) if away_ratings else None,
            "away_team_std_rating": np.std(away_ratings) if away_ratings else None,
            "away_team_avg_ADR": np.mean(away_adrs) if away_adrs else None,
            "away_team_std_ADR": np.std(away_adrs) if away_adrs else None,
            "away_team_avg_Swing": np.mean(away_swings) if away_swings else None,
            "away_team_std_Swing": np.std(away_swings) if away_swings else None,
            "maps_played": maps_played,
            "home_maps_won": maps_won,
            "away_maps_won": maps_lost,
            "home_maps_picked": maps_picked_by_home,
            "map_avg_score_diff": avg_score_diff,
            "home_map_winrate": map_winrate,
            "maps_json": map_json,
            "source_url": match_url
        })

    except Exception as e:
        print(f"Hiba a scraperben: {e}")

    driver.quit()
    return pd.DataFrame(data)


# Példa futtatás
url = "https://www.hltv.org/matches/2380078/mouz-vs-vitality-esl-pro-league-season-21"
df_h2h = scrape_head_to_head(url)
display(df_h2h)


,home_team,away_team,wins_home,wins_away,overtimes,total_non_overtime,home_win_rate,home_team_avg_rating,home_team_std_rating,home_team_avg_ADR,...,away_team_avg_Swing,away_team_std_Swing,maps_played,maps_won,maps_lost,maps_picked_by_home,map_avg_score_diff,map_winrate,maps_json,source_url
0,MOUZ,Vitality,5,9,1,14,0.3571,0.87,0.082704,68.46,...,2.706,1.21791,3,0,3,1,-7.0,0.0,"[{""map_name"": ""Dust2"", ""played"": true, ""home_w...",https://www.hltv.org/matches/2380078/mouz-vs-v...


In [ ]:
# Team history

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime as dt
import time

def scrape_team_results_summary(team_id):
    url = f"https://www.hltv.org/results?team={team_id}"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

    time.sleep(2)
    today = pd.Timestamp.now().normalize()

    matches = []

    try:
        results_holder = driver.find_element(By.CLASS_NAME, "results-holder")
        sublists = results_holder.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Talált results-sublist blokkok: {len(sublists)}")

        for sublist in sublists:
            # --- dátum fejléc ---
            headline_el = sublist.find_element(By.CLASS_NAME, "standard-headline")
            headline_text = headline_el.text.strip().replace("Results for", "").strip()
            date = pd.to_datetime(headline_text, errors='coerce')

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    # --- score ---
                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    # --- winner ---
                    team1_html = tds[0].get_attribute("innerHTML")
                    team2_html = tds[2].get_attribute("innerHTML")

                    if "team-won" in team1_html:
                        winner = team1_name
                    elif "team-won" in team2_html:
                        winner = team2_name
                    else:
                        winner = None

                    # --- target team name detektálás ---
                    is_target_team1 = str(team_id) in match_url and team1_name != ""  # fallback
                    is_target_team2 = str(team_id) in match_url and team2_name != ""  # fallback

                    # egyszerűbb: bármelyik oldalon is van a csapat, figyeljük, hogy nyert-e
                    target_team_name = team1_name if (score1 > score2) or ("team-won" in team1_html) else team2_name
                    result = "win" if score1 > score2 else "loss"

                    matches.append({
                        "date": date,
                        "team1": team1_name,
                        "team2": team2_name,
                        "score1": score1,
                        "score2": score2,
                        "winner": winner,
                        "result": result
                    })

                except Exception as e:
                    print(f"⚠️ Hiba meccs feldolgozásnál: {e}")
                    continue

        driver.quit()

        # --- DataFrame feldolgozás ---
        df = pd.DataFrame(matches)
        if df.empty:
            print("⚠️ Nincs adat!")
            return pd.DataFrame()

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).sort_values("date", ascending=False)
        df["days_ago"] = (today - df["date"]).dt.days

        # --- winrate számítás ---
        def winrate(df):
            if len(df) == 0:
                return None
            return round((df["result"] == "win").sum() / len(df), 4)

        last_30d = df[df["days_ago"] <= 30]
        last_90d = df[df["days_ago"] <= 90]
        last_7d = df[df["days_ago"] <= 7]

        last_30d_winrate = winrate(last_30d)
        last_90d_winrate = winrate(last_90d)
        matches_last_7d = len(last_7d)

        last_match_date = df["date"].max()
        days_since_last_match = int((today - last_match_date).days) if pd.notna(last_match_date) else None

        # --- current streak ---
        current_streak = 0
        if len(df) > 0:
            last_results = df["result"].tolist()
            last_outcome = last_results[0]
            for r in last_results:
                if r == last_outcome:
                    current_streak += 1
                else:
                    break
            if last_outcome == "loss":
                current_streak *= -1  # negatív streak veszteség esetén

        df_summary = pd.DataFrame([{
            "team_id": team_id,
            "scrape_date": today,
            "last_30d_winrate": last_30d_winrate,
            "last_90d_winrate": last_90d_winrate,
            "matches_last_7d": matches_last_7d,
            "days_since_last_match": days_since_last_match,
            "current_streak": current_streak,
            "total_matches_found": len(df),
            "source_url": url
        }])

        print(f"✅ Összesen {len(df)} meccs feldolgozva a csapatnál.")
        return df_summary

    except Exception as e:
        print(f"❌ Hiba a scraperben: {e}")
        driver.quit()
        return pd.DataFrame()


# --- Példa futtatás ---
team_id = 5378  # Virtus.pro
df_team_summary = scrape_team_results_summary(team_id)
display(df_team_summary)


# OddsPortal

In [ ]:
# Tournaments

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_oddsportal_cs_links():
    url = "https://www.oddsportal.com/results/#esports"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    # várjuk, amíg betöltődik legalább 1 tournament link
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")))

    time.sleep(2)  # kis extra wait, hogy minden JS lefusson

    links = driver.find_elements(By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")

    data = []
    for link in links:
        try:
            text = link.text.strip()
            href = link.get_attribute("href")
            if text.startswith("Counter-Strike "):
                name = text.replace("Counter-Strike ", "").strip()
                data.append({
                    "Name": name,
                    "url": href
                })
        except Exception as e:
            print(f"⚠️ Hiba egy link feldolgozásánál: {e}")
            continue

    driver.quit()
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} Counter-Strike esemény található az OddsPortalon.")
    return df


# --- Példa futtatás ---
df_oddsportal = scrape_oddsportal_cs_links()
display(df_oddsportal)


In [ ]:
# Odds of tournament

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re, time
from datetime import datetime

def scrape_oddsportal_fixed(event_url, headless=False):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0")
    
    driver = webdriver.Chrome(options=opts)
    driver.get(event_url)
    wait = WebDriverWait(driver, 20)
    
    all_data = []
    page = 1
    
    while True:
        print(f"🔍 Oldal {page} feldolgozása...")
        
        # Scroll, hogy minden betöltsön
        last_height = driver.execute_script("return document.body.scrollHeight")
        for _ in range(5):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.5)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        time.sleep(2)
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.eventRow")))

        event_rows = driver.find_elements(By.CSS_SELECTOR, "div.eventRow")
        current_date = None

        for event in event_rows:
            # dátum keresése
            date_found = False
            
            # Dátum keresése az event teljes szövegében
            date_text = event.text.strip()
            if any(month in date_text for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                lines = date_text.split('\n')
                for line in lines:
                    if any(month in line for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                        # Dátum formázása - eltávolítjuk a " -" utáni részt
                        clean_date = line.split(' -')[0].strip()
                        current_date = clean_date
                        
                        # Dátum átalakítása YYYY-MM-dd formátumba
                        try:
                            date_obj = datetime.strptime(current_date, '%d %b %Y')
                            current_date = date_obj.strftime('%Y-%m-%d')
                        except ValueError:
                            # Ha nem sikerül átalakítani, marad az eredeti
                            pass
                        
                        date_found = True
                        break
            
            # Ha dátum sor, akkor tovább
            if date_found:
                continue

            # Ha nincs dátum, de van game-row, akkor meccs sor
            try:
                game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
            except:
                continue

            # csapatnevek
            participants = game_row.find_elements(By.CSS_SELECTOR, "a[title]")
            if len(participants) < 2:
                continue

            home_team = participants[0].get_attribute("title").strip()
            away_team = participants[1].get_attribute("title").strip()

            # oddsok
            odds_blocks = event.find_elements(By.CSS_SELECTOR, "div[data-testid^='odd-container'] p")
            odds = []
            for p in odds_blocks:
                try:
                    odds_text = p.text.strip().replace(",", ".")
                    if re.match(r"^\d+(\.\d+)?$", odds_text):
                        odds.append(float(odds_text))
                except:
                    continue
            
            home_odds = odds[0] if len(odds) > 0 else None
            away_odds = odds[1] if len(odds) > 1 else None

            all_data.append({
                "Date": current_date,
                "home_team": home_team,
                "away_team": away_team,
                "home_odds": home_odds,
                "away_odds": away_odds
            })

        # Következő oldal ellenőrzése
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, f"a.pagination-link[data-number='{page + 1}']")
            if next_button.is_enabled():
                print(f"➡️ Következő oldal: {page + 1}")
                driver.execute_script("arguments[0].click();", next_button)
                page += 1
                time.sleep(3)  # Várakozás az oldal betöltésére
                continue
            else:
                break
        except:
            # Ha nincs következő oldal, kilépünk
            break

    driver.quit()
    
    df = pd.DataFrame(all_data).drop_duplicates(subset=["home_team","away_team","home_odds","away_odds"])
    print(f"✅ Összesen {len(df)} meccs feldolgozva {page} oldalról.")
    return df

# Teszt
url = "https://www.oddsportal.com/esports/counter-strike/counter-strike-the-perfect-world-shanghai-major/results/"
df = scrape_oddsportal_fixed(url, headless=False)
display(df)